# Car Price Prediction

**Tabular Regression Project** — Predict used car selling prices.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Used Car Data (301 rows × 9 columns)
Target: `Selling_Price` (in lakhs INR)

## 2 · Project Overview

We predict the **selling price of used cars** based on features like year, present price, kilometers driven, fuel type, seller type, and transmission. With only 301 rows, this is a small dataset where feature engineering and model selection matter more than model complexity.

## 3 · Learning Objectives

1. Handle categorical features (Fuel_Type, Seller_Type, Transmission).
2. Engineer age-based features from the year.
3. Compare models on small tabular data.
4. Understand used car depreciation patterns.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict the **Selling_Price** of a used car given its age, original price, mileage, fuel type, seller type, and transmission. This helps dealers price inventory and buyers assess fair value.

## 5 · Why This Project Matters

- **Used car pricing** is a huge market — accurate predictions save buyers and sellers money.
- Teaches depreciation modeling — a real-world business concept.
- Small dataset forces careful feature engineering.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 301 |
| **Columns** | 9 |
| **Features** | Car_Name, Year, Present_Price, Kms_Driven, Fuel_Type, Seller_Type, Transmission, Owner |
| **Target** | Selling_Price (lakhs INR) |
| **Categorical** | Fuel_Type, Seller_Type, Transmission |
| **File** | `car_data.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle / CarDekho dataset.
- **License**: Public / educational use.
- **Limitations**: Indian market data — prices in lakhs INR, not USD.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Selling_Price'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'car_data.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')
print(f'Target mean: {df[TARGET].mean():.2f}')
print(f'\nFuel types: {df["Fuel_Type"].value_counts().to_dict()}')
print(f'Seller types: {df["Seller_Type"].value_counts().to_dict()}')
print(f'Transmission: {df["Transmission"].value_counts().to_dict()}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df['Present_Price'], df[TARGET], alpha=0.5)
axes[0,0].set_xlabel('Present Price'); axes[0,0].set_ylabel(TARGET)
axes[0,0].set_title('Present Price vs Selling Price')

axes[0,1].scatter(df['Kms_Driven'], df[TARGET], alpha=0.5)
axes[0,1].set_xlabel('Kms Driven'); axes[0,1].set_ylabel(TARGET)
axes[0,1].set_title('Kms Driven vs Selling Price')

df.boxplot(column=TARGET, by='Fuel_Type', ax=axes[1,0])
axes[1,0].set_title('Price by Fuel Type')
plt.suptitle('')

df.boxplot(column=TARGET, by='Transmission', ax=axes[1,1])
axes[1,1].set_title('Price by Transmission')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing Strategy

- Drop `Car_Name` (too many unique values for 301 rows).
- Encode `Fuel_Type`, `Seller_Type`, `Transmission` with label encoding.
- Engineer `car_age` from Year.

In [ ]:
# Drop Car_Name
df = df.drop(columns=['Car_Name'])

# Encode categoricals
from sklearn.preprocessing import LabelEncoder
for col in ['Fuel_Type', 'Seller_Type', 'Transmission']:
    df[col] = LabelEncoder().fit_transform(df[col])

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
# Car age
df['car_age'] = 2026 - df['Year']
df['depreciation_ratio'] = df[TARGET] / (df['Present_Price'] + 0.01)
df['kms_per_year'] = df['Kms_Driven'] / (df['car_age'] + 1)

# Drop Year (replaced by car_age)
df = df.drop(columns=['Year'])

X = df.drop(columns=[TARGET, 'depreciation_ratio'])  # depreciation_ratio would be leakage
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Present_Price** (original showroom price) is the strongest predictor — cars retain a fraction of their original value.
- **Car_age** (derived from Year) captures depreciation directly.
- **Fuel_Type** and **Transmission** affect resale value — diesel and automatic tend to retain value better.
- Dealers can use this model to instantly price incoming trade-ins.

## 26 · Limitations

- Only 301 rows — a larger dataset would improve accuracy.
- Indian market-specific — prices in lakhs INR.
- No make/model encoding (Car_Name dropped due to high cardinality).
- No condition/accident history features.

## 27 · How to Improve This Project

1. Target-encode Car_Name (brand/model level).
2. Add make and model as separate features.
3. Include vehicle condition, service history.
4. Collect more data points (10K+ rows).
5. Try log-transforming the target.

## 28 · Production Considerations

- Real-time pricing requires up-to-date market data.
- Model needs retraining as market conditions change.
- Should include confidence intervals for pricing decisions.
- API integration with car listing platforms.

## 29 · Common Mistakes

1. **Including depreciation_ratio as feature**: It uses the target — leakage!
2. **Not dropping Car_Name**: 301 rows can't learn 100+ car names.
3. **Using Year directly instead of car_age**: Less interpretable.
4. **Not encoding categoricals**: Fuel_Type is a string, not ordinal.

## 30 · Mini Challenge / Exercises

1. Extract the brand from Car_Name and use as a feature.
2. Try log(Selling_Price) as target.
3. Add interaction: Present_Price × car_age.
4. Compare model accuracy for expensive vs. cheap cars.

## 31 · Final Summary / Key Takeaways

- Used car prices are predictable from a few key features.
- Present price and car age dominate — depreciation is the main driver.
- Boosting models work well even on 301 rows.
- Feature engineering (car_age, kms_per_year) adds meaningful signal.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['Baseline_LR'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')